# SHAP-интерпретация финалистов

Объясняем, какие признаки и как двигают предсказанный риск рецидива у каждого финалиста: четыре семейства (логистическая регрессия, случайный лес, XGBoost, CatBoost) на двух наборах признаков (significant 10 и no_collinear 25). Это нужно и для статьи (содержательная проверка модели), и для сервиса (объяснение конкретного прогноза врачу).

Способ расчета SHAP зависит от модели: для логистической регрессии - линейный SHAP, для случайного леса и XGBoost - TreeExplainer по преобразованной матрице (после импутации и one-hot), для CatBoost - встроенные ShapValues на нативной матрице с категориями. Логика в `src/interpret.py`, гиперпараметры из ноутбука 07.

При коррелирующих признаках SHAP делит вклад между ними, поэтому глобальную важность сверяем с Таблицей 1 (ноутбук 03) и клинической логикой, а не читаем буквально.

In [ ]:
import sys, pathlib, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import Image, display
from src import interpret, config

# Считает SHAP по всем восьми финалистам, сохраняет фигуры и сводную важность.
allimp = interpret.build_all()
print('\nГотово. Сводная важность:', allimp.shape)

## Топ признаков по финалистам

Топ-6 признаков по среднему |SHAP| для каждого финалиста. У моделей с one-hot категориальный признак разбит на уровни (например ХБП, вид инсулинотерапии), у CatBoost категория идет одним признаком.

In [ ]:
for (model, fset), g in allimp.groupby(['модель', 'набор'], sort=False):
    print(f'=== {model}, {fset} ===')
    display(g.head(6)[['признак', 'средний_|SHAP|']].reset_index(drop=True))

## Глобальная важность: beeswarm и столбчатые графики

Beeswarm: каждая точка - пациент, положение по горизонтали - вклад признака в риск (правее - сильнее в плюс), цвет - значение признака (красный высокое, синий низкое). Признаки отсортированы по силе влияния. Столбчатый график - средний |SHAP| без знака.

In [ ]:
for model, fset in interpret.FINALISTS:
    tag = f'{model}_{fset}'
    print(f'=== {model}, {fset} ===')
    for kind in ['beeswarm', 'bar']:
        f = config.FIGURES_DIR / f'shap_{kind}_{tag}.png'
        if f.exists():
            display(Image(str(f)))

## Сверка с Таблицей 1 и клинической логикой

Ведущие по SHAP признаки совпадают со значимыми из Таблицы 1 у всех восьми финалистов. Устойчивое ядро, попадающее в топ независимо от модели и набора, это вид инсулинотерапии на момент ДКА, суточная доза инсулина, стадии ХБП (С и А), HbA1c, ЛПВП и длительность СД. Вторым эшелоном идут ретинопатия, диабетическая нейропатия и алкоголь за сутки до эпизода. Все девять предикторов, отобранных по Таблице 1, проявляются и здесь, то есть статистический отбор и важность по модели указывают на одни и те же факторы.

CatBoost дает самый читаемый рейтинг, так как работает с категориями целиком, без разбиения на уровни: вид инсулинотерапии (средний |SHAP| 0.58-0.64) с большим отрывом, затем суточная доза инсулина (0.24-0.31), ХБП-А, тип СД, глюкоза при поступлении, ЛПВП, диабетическая нейропатия и HbA1c. У моделей с one-hot тот же сигнал распределен по уровням категории (например ХБП, С_0 и ХБП, С_1 у логистической регрессии), поэтому отдельный уровень выглядит слабее, чем признак целиком.

На наборе no_collinear подключаются лабораторные показатели поступления (глюкоза, натрий, калий, мочевина, общий холестерин), но их вклад заметно меньше, чем у инсулинотерапии, дозы инсулина и ХБП. Это согласуется с клинической логикой: тяжесть текущего эпизода и острые электролиты меньше говорят о вероятности повторного ДКА, чем хронические факторы (контроль гликемии, осложнения, режим инсулинотерапии).

Направление вкладов читается по beeswarm (цвет точки против ее положения) и согласуется с отношениями шансов из Таблицы 1. Оговорка про корреляции: SHAP делит вклад между связанными признаками (липиды между собой, стадии ХБП), поэтому абсолютную величину для отдельного коррелирующего признака не интерпретируем буквально, опираемся на группу и на Таблицу 1.

## Локальные объяснения: waterfall

Waterfall показывает, как факторы конкретного пациента поднимают (красные) и опускают (синие) риск относительно базового уровня. Для каждого финалиста берем пациента самого высокого и самого низкого предсказанного риска. Это прообраз объяснения в сервисе: врач видит, что именно подняло риск у пациента.

In [ ]:
for model, fset in interpret.FINALISTS:
    tag = f'{model}_{fset}'
    print(f'=== {model}, {fset} ===')
    for label in ['high', 'low']:
        f = config.FIGURES_DIR / f'shap_waterfall_{tag}_{label}.png'
        if f.exists():
            display(Image(str(f)))

## Вывод

Интерпретация согласована между финалистами и со статистикой. Ведущие факторы риска повторного ДКА по SHAP, общие для всех моделей, это режим инсулинотерапии, суточная доза инсулина, стадии ХБП, HbA1c, ЛПВП и длительность СД, то есть хронические характеристики течения диабета и его осложнений, а не острые показатели текущего эпизода. Совпадение с Таблицей 1 повышает доверие к моделям: они опираются на клинически осмысленные признаки, а не на артефакты.

Для сервиса подходят все восемь финалистов: глобальная важность объясняет, на что смотрит модель в целом, а waterfall дает разбор конкретного пациента (какие факторы подняли и опустили его риск). CatBoost удобнее для объяснения врачу, так как категориальные осложнения и режим инсулинотерапии идут целыми признаками, без дробления на уровни.

Оговорка остается прежней: при коррелирующих признаках SHAP распределяет вклад между ними, поэтому в интерфейсе показываем группу связанных факторов и сверяемся с Таблицей 1, а не приписываем весь эффект одному показателю.